# ECG-DSP-Toolkit: guided walkthrough

Run the cells top to bottom. Each section calls the library in `src/` (the same code the scripts and
tests use) and ends with **questions you should be able to answer in an interview**.

If the MIT-BIH data has not been downloaded (`python data/download_data.py`), the notebook falls back to a
**synthetic** ECG and says so in every plot title. Synthetic output is for checking that the code runs,
not for drawing conclusions about real ECGs.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

from src import fft, windows, psd, fir, iir, noise, metrics, peaks, pipeline, analysis
from src import io as ecgio

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42

REAL = ecgio.data_available()
if REAL:
    rec = ecgio.load_record("100", duration_s=120)
    TAG = ""
else:
    rec = ecgio.synthetic_record(duration_s=120, seed=SEED)
    TAG = "[SYNTHETIC] "
x, fs = rec["signal"], rec["fs"]
print(f"{TAG}record {rec['name']}: {len(x)} samples, fs = {fs:g} Hz, {len(rec['ann_sample'])} annotated beats")

## 1. FFT from scratch

`X[k] = E[k] + W^k O[k]`, `X[k+N/2] = E[k] - W^k O[k]`. Below we verify both implementations and look at the effect of zero padding.

In [ ]:
seg = x[:3000]                      # 3000 samples: not a power of two
X = fft.fft(seg)                    # zero-padded to 4096
print("padded length:", len(X))
print("iterative == numpy :", np.allclose(X, np.fft.fft(seg, n=4096)))
print("recursive == numpy :", np.allclose(fft.fft(seg, 'recursive'), np.fft.fft(seg, n=4096)))
print("ifft(fft(x)) == x  :", np.allclose(fft.ifft(X)[:3000].real, seg))

f = fft.rfftfreq(len(X), fs)
plt.plot(f, 20*np.log10(np.abs(X[:len(f)]) + 1e-12))
plt.title(TAG + "ECG magnitude spectrum from our FFT"); plt.xlabel("Hz"); plt.ylabel("dB"); plt.show()

**Check yourself:** Why does the iterative version need a bit-reversal permutation? What does zero padding change
(and not change) about the spectrum? Why is the cost `O(N log N)`?

## 2. Windows and leakage

In [ ]:
N, nfft = 512, 8192
t = np.arange(N) / fs
tones = [(60.3, 1.0), (68.5, 0.003)]            # a strong tone and one 50 dB weaker
sig = sum(a*np.sin(2*np.pi*f0*t) for f0, a in tones) + 1e-4*np.random.default_rng(SEED).standard_normal(N)
for name in ["rectangular", "hann", "blackman"]:
    w = windows.get_window(name, N, sym=False)
    S = fft.fft(np.pad(sig*w, (0, nfft-N)), pad=False)[:nfft//2+1]
    plt.plot(np.arange(nfft//2+1)*fs/nfft, 20*np.log10(np.abs(S)/(w.sum()/2)+1e-12), label=name, lw=1)
plt.xlim(50, 75); plt.ylim(-120, 5); plt.legend(); plt.xlabel("Hz"); plt.ylabel("dB")
plt.title("Which window reveals the weak tone at 68.5 Hz?"); plt.show()

**Check yourself:** Why do the rectangular-window sidelobes hide the weak tone? What do you give up by switching to Blackman?

## 3. Welch PSD (own implementation vs scipy)

In [ ]:
f1, P1 = psd.welch(x, fs, nperseg=1024)
f2, P2 = signal.welch(x, fs, nperseg=1024, nfft=1024)
print("max relative difference:", np.max(np.abs(P1-P2)/np.abs(P2)))
plt.semilogy(f1, P1); plt.title(TAG + "Welch PSD of the ECG"); plt.xlabel("Hz"); plt.ylabel("mV^2/Hz"); plt.show()

**Check yourself:** Why does averaging segments reduce variance, and what does it cost? Why divide by `fs * sum(w**2)`?

## 4. FIR design: order from the transition width

In [ ]:
for tw in (0.5, 1.0, 2.0):
    print(f"Hamming, transition {tw} Hz -> {fir.estimate_numtaps(tw, fs, 'hamming')} taps")
h = fir.design_bandpass_fir(0.5, 40.0, fs, transition_hz=0.5)
M = len(h) - 1
print("taps:", len(h), "| delay:", M/2/fs, "s | symmetric:", np.allclose(h, h[::-1]))
ff, H = analysis.freqz_fir(h, fs)
plt.plot(ff, analysis.mag_db(H)); plt.xlim(0, 100); plt.ylim(-100, 5)
plt.title("FIR magnitude response"); plt.xlabel("Hz"); plt.ylabel("dB"); plt.show()

**Check yourself:** Where does `3.3 fs / delta_f` come from? Why is the filter so long? What does symmetry buy you?

## 5. IIR, bilinear transform, causal vs zero-phase

In [ ]:
bank = pipeline.build_filter_bank(fs, mains_hz=60.0)
lo, hi = 15 * int(fs), 15 * int(fs) + int(8 * fs)
x0 = x - x.mean()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(lo, hi)/fs, x0[lo:hi], color="0.75", lw=3, label="input")
for m in ("FIR", "IIR (causal)", "IIR (filtfilt)"):
    ax.plot(np.arange(lo, hi)/fs, bank.apply(m, x)[lo:hi], lw=1, label=m)
ax.legend(); ax.set_title(TAG + "Same beats through three filters"); ax.set_xlabel("s"); plt.show()

**Check yourself:** Why pre-warp? Why SOS instead of one (b, a) polynomial? What does `filtfilt` do to the magnitude response and why can't it run in real time?

## 6. One denoising experiment end to end

In [ ]:
kind, snr = "ma", 6.0
if REAL:
    nz, _ = ecgio.load_noise(kind, n_samples=len(x), offset=1000)
else:
    nz = noise.synthetic_noise(kind, len(x), fs, seed=SEED)
noisy, scaled, alpha = noise.mix_at_snr(x, nz, snr)
print(f"alpha = {alpha:.4f}; measured SNR = {noise.measured_snr_db(x, scaled):.6f} dB (target {snr})")

cut = metrics.trim_slice(len(x), fs, 5.0)
ref = rec["ann_sample"]; ref = ref[(ref >= cut.start) & (ref < cut.stop)]
for m in ("none",) + pipeline.METHODS:
    y = noisy if m == "none" else bank.apply(m, noisy)
    det = peaks.detect_r_peaks(y, fs); det = det[(det >= cut.start) & (det < cut.stop)]
    sc = metrics.peak_scores(*metrics.match_peaks(ref, det, int(0.15*fs)))
    print(f"{m:<30s} SNR_out {metrics.snr_db(x[cut], y[cut]):6.2f} dB  RMSE {metrics.rmse(x[cut], y[cut]):.3f} mV  "
          f"Se {100*sc['se']:5.1f}%  PPV {100*sc['ppv']:5.1f}%")

**Exercises**

1. Change `kind` to `"bw"` and `"em"`; which filter wins in each case, and why?
2. Set the notch to 50 Hz (`mains_hz=50.0`) and add `noise.mains_interference(len(x), fs, 50.0)` at 12 dB.
3. Re-design the FIR with `transition_hz=2.0`. What do you lose and gain?
4. Try `iir_order=1..5` in `build_filter_bank`. When does the causal filter start distorting the QRS?